# 이슈 #38 구조환경지표 원자료 기반 전수검증

## 목적과 현재 범위

- `configs/issue38_structural_indicators.yaml`을 기준으로 구조환경지표 21개의 가공값과 원자료 연결 상태를 점검한다.
- 현재 상세 검증이 완료된 지표는 `2-1.1. 보육시설 보급률`이다. 나머지 지표의 산식 검증은 후속 작업이다.
- 기존 가공값은 `어린이집 수 ÷ 0–4세 인구 × 100`과 전부 일치했으며, 명세상 올바른 산식은 `어린이집 정원 ÷ 0–4세 인구 × 100`이다.
- 이 노트북은 수정값을 메모리에 생성할 뿐 원본 또는 기존 가공 파일을 덮어쓰지 않는다.

## 1. 실행 환경과 검증 명세 확인

저장소 루트를 찾고 필수 라이브러리를 불러온 뒤, 검증 명세와 구조환경 원자료 경로가 존재하는지 확인한다. YAML의 필수 구역, 지표 수, 지표 ID 중복 여부도 검사한다. 정상 출력은 이슈 `#38`, 검증 대상 `21개`, 고유 ID `21개`이다.

In [11]:
import sys

import numpy as np
import openpyxl
import pandas as pd
import yaml

from pathlib import Path

repo_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / ".git").exists()), None)

if repo_root is None:
    raise FileNotFoundError("현재 실행 위치의 상위 경로에서 Git 저장소를 찾지 못했습니다.")

print(f"저장소 루트: {repo_root}")

저장소 루트: d:\University\yumocha\yumocha


In [12]:
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / ".git").exists():
            return path

    raise FileNotFoundError("현재 실행 위치의 상위 경로에서 Git 저장소를 찾지 못했습니다.")


repo_root = find_repo_root(Path.cwd())

In [13]:
config_path = repo_root / "configs" / "issue38_structural_indicators.yaml"
raw_root = repo_root / "data" / "raw" / "2. 구조환경지수 원데이터"

missing_paths = [path for path in (config_path, raw_root) if not path.exists()]
if missing_paths:
    raise FileNotFoundError("다음 입력 경로를 찾지 못했습니다:\n" + "\n".join(map(str, missing_paths)))

print(f"검증 명세: {config_path.relative_to(repo_root)}")
print(f"구조환경 원자료: {raw_root.relative_to(repo_root)}")

검증 명세: configs\issue38_structural_indicators.yaml
구조환경 원자료: data\raw\2. 구조환경지수 원데이터


In [14]:
with config_path.open(encoding="utf-8") as file:
    manifest = yaml.safe_load(file)

required_sections = {"manifest_version", "issue", "scope", "rules", "local", "drive", "indicators"}
missing_sections = required_sections - manifest.keys()
if missing_sections:
    raise KeyError(f"YAML 필수 구역이 없습니다: {sorted(missing_sections)}")

indicators = manifest["indicators"]
if not isinstance(indicators, list) or not all(isinstance(indicator, dict) for indicator in indicators):
    raise TypeError("indicators는 지표 딕셔너리로 구성된 리스트여야 합니다.")

indicator_ids = [indicator.get("id") for indicator in indicators]
if len(indicators) != manifest["scope"]["completed_indicators_to_validate"]:
    raise ValueError("검증 대상 지표 수가 scope의 선언값과 다릅니다.")
if None in indicator_ids or len(indicator_ids) != len(set(indicator_ids)):
    raise ValueError("지표 ID가 없거나 중복됐습니다.")

print(
    f"명세 버전: {manifest['manifest_version']} | 이슈: #{manifest['issue']} | "
    f"검증 대상: {len(indicators)}개 | 고유 ID: {len(set(indicator_ids))}개"
)

명세 버전: 2 | 이슈: #38 | 검증 대상: 21개 | 고유 ID: 21개


## 2. 명세와 로컬 원자료 연결 확인

YAML에 기록된 가공 파일·원자료 파일명을 로컬 원자료 폴더와 연결한다. 같은 이름의 파일이 여러 개면 SHA-256 해시로 실제 내용까지 비교한다. 현재 누락 파일은 없지만, 주민등록인구 파일은 같은 이름의 서로 다른 내용이 있어 지표에 맞는 파일을 명시적으로 선택해야 한다.

In [15]:
local_files: list[Path] = sorted(
    (path for path in raw_root.rglob("*") if path.is_file()),
    key=lambda path: path.as_posix().casefold(),
)

if not local_files:
    raise FileNotFoundError(f"원자료 폴더가 비어 있습니다: {raw_root}")

paths_by_name: dict[str, list[Path]] = {}

for path in local_files:
    paths_by_name.setdefault(path.name, []).append(path)

mapping_records: list[dict[str, str]] = []

for indicator in indicators:
    mapping_records.append(
        {
            "indicator_id": indicator["id"],
            "mapping_type": "processed",
            "role": "processed",
            "drive_id": indicator["processed"]["id"],
            "file_name": indicator["processed"]["file_name"],
        }
    )

    for source in indicator["raw_sources"]:
        mapping_records.append(
            {
                "indicator_id": indicator["id"],
                "mapping_type": "raw",
                "role": source["role"],
                "drive_id": source["id"],
                "file_name": source["file_name"],
            }
        )

resolution_rows: list[dict[str, object]] = []

for record in mapping_records:
    matched_paths = paths_by_name.get(record["file_name"], [])

    resolution_rows.append(
        {
            **record,
            "match_count": len(matched_paths),
            "local_paths": " | ".join(
                str(path.relative_to(repo_root)) for path in matched_paths
            ),
        }
    )

file_resolution = pd.DataFrame(resolution_rows)

missing_file_mappings = file_resolution[
    file_resolution["match_count"] == 0
].copy()

ambiguous_file_mappings = file_resolution[
    file_resolution["match_count"] > 1
].copy()

problem_file_mappings = file_resolution[
    file_resolution["match_count"] != 1
].copy()

print(
    f"로컬 탐색 파일: {len(local_files)}개 | "
    f"매핑 참조: {len(file_resolution)}건 | "
    f"정확히 1개 일치: {(file_resolution['match_count'] == 1).sum()}건 | "
    f"누락: {len(missing_file_mappings)}건 | "
    f"다중 일치: {len(ambiguous_file_mappings)}건"
)

if not problem_file_mappings.empty:
    print("\n확인이 필요한 매핑:")
    print(
        problem_file_mappings[
            ["indicator_id", "mapping_type", "role", "file_name", "match_count", "local_paths"]
        ].to_string(index=False)
    )

로컬 탐색 파일: 210개 | 매핑 참조: 65건 | 정확히 1개 일치: 49건 | 누락: 0건 | 다중 일치: 16건

확인이 필요한 매핑:
               indicator_id mapping_type                role                                        file_name  match_count                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [16]:
import hashlib


def calculate_sha256(path: Path) -> str:
    hasher = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(chunk)

    return hasher.hexdigest()


ambiguous_names = sorted(
    str(file_name)
    for file_name in ambiguous_file_mappings["file_name"].unique()
)

hash_rows: list[dict[str, object]] = []

for file_name in ambiguous_names:
    for path in paths_by_name[file_name]:
        hash_rows.append(
            {
                "file_name": file_name,
                "local_path": path.relative_to(raw_root).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": calculate_sha256(path),
            }
        )

duplicate_file_hashes = pd.DataFrame(hash_rows)

duplicate_content_summary = (
    duplicate_file_hashes.groupby("file_name", as_index=False).agg(candidate_count=("local_path", "size"), distinct_size_count=("size_bytes", "nunique"), distinct_hash_count=("sha256", "nunique"),)
)

duplicate_content_summary["content_identical"] = (
    duplicate_content_summary["distinct_hash_count"] == 1
)

different_content_files = duplicate_content_summary[
    ~duplicate_content_summary["content_identical"]
].copy()

print(
    f"중복 파일명: {len(duplicate_content_summary)}개 | "
    f"후보 파일: {len(duplicate_file_hashes)}개 | "
    f"내용 동일: {duplicate_content_summary['content_identical'].sum()}개 | "
    f"내용 다름: {len(different_content_files)}개"
)

if not different_content_files.empty:
    print("\n내용이 다른 중복 파일:")
    display(different_content_files)
    display(
        duplicate_file_hashes[
            duplicate_file_hashes["file_name"].isin(
                different_content_files["file_name"]
            )
        ]
    )

중복 파일명: 8개 | 후보 파일: 25개 | 내용 동일: 7개 | 내용 다름: 1개

내용이 다른 중복 파일:


,file_name,candidate_count,distinct_size_count,distinct_hash_count,content_identical
3,2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv.xlsx,6,4,4,False


,file_name,local_path,size_bytes,sha256
11,2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv.xlsx,3. 제주여성가족연구원 지표별 측정값(raw)/2-1.1. 보육시설 보급률 원자료/...,517671,b12cb901b7df210451c983699528c89ad18f9a8aa1ead2...
12,2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv.xlsx,3. 제주여성가족연구원 지표별 측정값(raw)/2-1.2. 방과후 돌봄시설 보급도 ...,491302,c6d75b98836377e276eab9a66e2348e37b1920f2224c70...
13,2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv.xlsx,3. 제주여성가족연구원 지표별 측정값(raw)/3-1.1. (대체)분만실 병상수 보...,517270,fa9c747b342e910e74bc5824859aba9599893d5c2b67c7...
14,2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv.xlsx,3. 제주여성가족연구원 지표별 측정값(raw)/3-1.2. (대체)소아청소년과 전문...,517270,fa9c747b342e910e74bc5824859aba9599893d5c2b67c7...
15,2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv.xlsx,"3. 제주여성가족연구원 지표별 측정값(raw)/3-2.1. 산후조리원 보급도, 3-...",517270,fa9c747b342e910e74bc5824859aba9599893d5c2b67c7...
16,2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv.xlsx,3. 제주여성가족연구원 지표별 측정값(raw)/3-3.1. 어린이 교통사고 발생률 ...,518146,fed52d095bea377d50695aadd879f3d557dd4ccfb65a7d...


## 3. 보육시설 보급률 입력 파일 확정

- `processed`: 검증할 기존 보육시설 보급률
- `numerator_active`: 올바른 분자인 어린이집 정원
- `denominator`: 0–4세 주민등록인구
- `numerator_excluded_facility_count`: 기존 오류 산식을 입증하기 위한 어린이집 수 비교자료이며, 수정 산식의 분자로 사용하지 않는다.

내용이 다른 동명 주민등록인구 파일 가운데 `2-1.1. 보육시설 보급률 원자료` 폴더의 파일을 분모로 고정한다.

In [17]:
#cell 8
population_path = raw_root / (
    "3. 제주여성가족연구원 지표별 측정값(raw)/"
    "2-1.1. 보육시설 보급률 원자료/"
    "2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv.xlsx"
)

if not population_path.is_file():
    raise FileNotFoundError(population_path)

local_path_overrides = {
    ("childcare_capacity_rate", "raw", "denominator"): population_path
}

In [18]:
# cell 9

childcare = next(
    item for item in indicators
    if item["id"] == "childcare_capacity_rate"
)

childcare_paths = {
    "processed": paths_by_name[childcare["processed"]["file_name"]][0],
    **{
        source["role"]: local_path_overrides.get(
            (childcare["id"], "raw", source["role"]),
            paths_by_name[source["file_name"]][0],
        )
        for source in childcare["raw_sources"]
    },
}

for role, path in childcare_paths.items():
    print(f"{role}: {path.relative_to(repo_root)}")

processed: data\raw\2. 구조환경지수 원데이터\3. 제주여성가족연구원 지표별 측정값(raw)\2-1.1. 보육시설 보급률.xlsx
numerator_active: data\raw\2. 구조환경지수 원데이터\3. 제주여성가족연구원 지표별 측정값(raw)\2-1.1. 보육시설 보급률 원자료\전국_어린이집_정현원_현황_20260724113247.csv.xlsx
denominator: data\raw\2. 구조환경지수 원데이터\3. 제주여성가족연구원 지표별 측정값(raw)\2-1.1. 보육시설 보급률 원자료\2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv.xlsx
numerator_excluded_facility_count: data\raw\2. 구조환경지수 원데이터\1. 제주여성가족연구원 지표별 원데이터\2-1. 돌봄여건\1. 보육시설 보급률\2016-2025_어린이집_현황_시·도__20260715191601.csv


## 4. 입력 자료 구조 확인

선택한 엑셀 파일의 시트명, 행·열 크기, 상단 값을 미리 확인한다. 이 단계에서는 값을 변환하지 않으며, 2016–2025년 지역별 정원과 0–4세 인구가 실제로 들어 있는지 확인한다.

In [ ]:
# cell 10

for role, path in childcare_paths.items():
    if path.suffix.lower() != ".xlsx":
        continue

    workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
    print(f"\n[{role}] {path.name}")

    for worksheet in workbook.worksheets:
        preview = list(worksheet.iter_rows(max_row=8, values_only=True))
        print(f"{worksheet.title}: {worksheet.max_row}행 × {worksheet.max_column}열")
        display(pd.DataFrame(preview))

    workbook.close()


[processed] 2-1.1. 보육시설 보급률.xlsx


2-1.1. 보육시설 보급률: 19행 × 12열


,0,1,2,3,4,5,6,7,8,9,10,11
0,지역,세부지표,2016.000000,2017.000000,2018.000000,2019.000000,2020.000000,2021.000000,2022.000000,2023.000000,2024.000000,2025.000000
1,전국,보육시설 보급률,1.863836,1.935343,1.984101,2.025395,2.108021,2.172275,2.166098,2.169508,2.153445,2.077639
2,서울,보육시설 보급률,1.691031,1.777450,1.851635,1.894288,1.985352,2.064710,2.070126,2.080145,2.075930,1.979006
3,부산,보육시설 보급률,1.466935,1.551139,1.614956,1.711698,1.825949,1.900248,1.903485,1.920117,1.903396,1.820668
4,대구,보육시설 보급률,1.494990,1.561200,1.576827,1.596483,1.708895,1.788809,1.860442,1.873687,1.886139,1.819249
5,인천,보육시설 보급률,1.711152,1.793656,1.855752,1.905036,2.017234,2.031458,1.983658,1.996302,1.981792,1.875218
6,광주,보육시설 보급률,1.893980,2.023730,2.061376,2.080398,2.198073,2.209531,2.211817,2.256141,2.294264,2.267541
7,대전,보육시설 보급률,2.322172,2.381291,2.396210,2.410901,2.485110,2.532723,2.434335,2.305505,2.258587,2.167126


보육시설 보급률 계산: 57행 × 13열


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,지역,항목,NaN,2016 년,2017 년,2018 년,2019 년,2020 년,2021 년,2022 년,2023 년,2024 년,2025 년
1,전국,총인구수[명],0-4세,2204271,2079115,1974244,1845122,1677023,1530469,1427590,1334588,1271776,1254501
2,서울,총인구수[명],0-4세,376575,350277,324470,300799,270481,244538,227619,213014,202897,202627
3,부산,총인구수[명],0-4세,132044,123780,117093,107963,97374,87778,81272,75360,71136,70194
4,대구,총인구수[명],0-4세,99198,93774,89103,82807,74200,66357,61222,57587,54874,54528
5,인천,총인구수[명],0-4세,130380,121874,115371,107557,96320,88754,85549,82753,81391,82977
6,광주,총인구수[명],0-4세,65365,61273,57971,53932,48770,45349,42499,38916,36090,34619
7,대전,총인구수[명],0-4세,68212,63201,58676,53424,47684,43471,41613,39731,38254,38115


...

,0,1,2,3,4,5,6,7,8,9,10
0,시도별,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
1,전국,41084,40238,39171,37371,35352,33246,30923,28954,27387,26064
2,서울특별시,6368,6226,6008,5698,5370,5049,4712,4431,4212,4010
3,부산광역시,1937,1920,1891,1848,1778,1668,1547,1447,1354,1278
4,대구광역시,1483,1464,1405,1322,1268,1187,1139,1079,1035,992
5,인천광역시,2231,2186,2141,2049,1943,1803,1697,1652,1613,1556
6,광주광역시,1238,1240,1195,1122,1072,1002,940,878,828,785
7,대전광역시,1584,1505,1406,1288,1185,1101,1013,916,864,826



[numerator_active] 전국_어린이집_정현원_현황_20260724113247.csv.xlsx
전국_어린이집_정현원_현황_20260724113247: 19행 × 12열


,0,1,2,3,4,5,6,7,8,9,10,11
0,시도별,항목,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
1,전국,정원,1767224,1756603,1732324,1686873,1628260,1557496,1476686,1401440,1340087,1285032
2,서울특별시,정원,270231,268100,263157,254538,245863,235682,223628,211248,202722,195742
3,부산광역시,정원,89658,89040,88222,86553,84008,80248,76146,71570,66235,62266
4,대구광역시,정원,75255,74696,72500,69390,66727,62125,59434,56462,53946,52133
5,인천광역시,정원,94314,93938,93888,91615,88002,84877,80896,78883,76831,75036
6,광주광역시,정원,62528,63161,61121,57756,55020,49556,46531,43399,41021,38868
7,대전광역시,정원,54514,53688,51025,47923,45160,42171,39145,36356,34983,34111



[denominator] 2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv.xlsx
2016-2025_행정구역(시도)별_1세별_주민등록인구_: 5509행 × 15열


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,행정구역(시군구)별,연령별,항목,단위,2026.06 월,2016 년,2017 년,2018 년,2019 년,2020 년,2021 년,2022 년,2023 년,2024 년,2025 년
1,전국,계,총인구수[명],명,51091769,51696216,51778544,51826059,51849861,51829023,51638809,51439038,51325329,51217221,51117378
2,전국,계,남자인구수[명],명,25418248,25827594,25855919,25866129,25864816,25841029,25746684,25636951,25565736,25498324,25436665
3,전국,계,여자인구수[명],명,25673521,25868622,25922625,25959930,25985045,25987994,25892125,25802087,25759593,25718897,25680713
4,전국,0세,총인구수[명],명,270633,393674,345786,317685,295132,265087,253946,244250,225958,235337,252212
5,전국,0세,남자인구수[명],명,138686,201541,178138,162950,151494,135661,130157,124860,115707,120438,129588
6,전국,0세,여자인구수[명],명,131947,192133,167648,154735,143638,129426,123789,119390,110251,114899,122624
7,전국,1세,총인구수[명],명,253098,441720,409814,361625,330970,304651,274633,264788,253595,234405,243765


### 비교용 어린이집 수 자료

기존 가공값이 어린이집 수를 사용했는지 확인하기 위해 CSV를 별도로 읽는다. 인코딩을 자동 판별하며, 현재 파일은 `cp949`, 18개 지역 × 10개 연도 구조이다.

In [20]:
# cell 11

facility_count_path = childcare_paths[
    "numerator_excluded_facility_count"
]

for encoding in ("utf-8-sig", "cp949", "euc-kr"):
    try:
        facility_count_raw = pd.read_csv(
            facility_count_path,
            encoding=encoding,
        )
        facility_count_encoding = encoding
        break
    except UnicodeDecodeError:
        continue
else:
    raise UnicodeError(
        f"CSV 인코딩을 확인하지 못했습니다: {facility_count_path}"
    )

print(f"인코딩: {facility_count_encoding}")
print(
    f"크기: {facility_count_raw.shape[0]}행 × "
    f"{facility_count_raw.shape[1]}열"
)
print(f"열 이름: {facility_count_raw.columns.tolist()}")

display(facility_count_raw.head(8))

인코딩: cp949
크기: 18행 × 11열
열 이름: ['시도별', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


,시도별,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,전국,41084,40238,39171,37371,35352,33246,30923,28954,27387,26064
1,서울특별시,6368,6226,6008,5698,5370,5049,4712,4431,4212,4010
2,부산광역시,1937,1920,1891,1848,1778,1668,1547,1447,1354,1278
3,대구광역시,1483,1464,1405,1322,1268,1187,1139,1079,1035,992
4,인천광역시,2231,2186,2141,2049,1943,1803,1697,1652,1613,1556
5,광주광역시,1238,1240,1195,1122,1072,1002,940,878,828,785
6,대전광역시,1584,1505,1406,1288,1185,1101,1013,916,864,826
7,울산광역시,895,881,868,842,790,720,656,612,569,535


## 5. 정원·시설 수·0–4세 인구 결합

지역명을 `서울특별시 → 서울`처럼 통일하고 세 자료를 `지역 × 연도` 긴 형식으로 변환한다. 주민등록인구는 0–4세의 `총인구수[명]`만 합산한다. 이후 세 자료를 일대일 결합하고, 18개 지역 × 10개 연도인 180행과 결측값 없음까지 검증한다. 결과 변수는 `childcare_comparison`이다.

In [21]:
# cell 12

years = [str(y) for y in range(2016, 2026)]
region_order = ["전국", "서울", "부산", "대구", "인천", "광주", "대전", "울산", "세종",
                "경기", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]

region_map = {r: r for r in region_order} | {
    "서울특별시": "서울", "부산광역시": "부산", "대구광역시": "대구",
    "인천광역시": "인천", "광주광역시": "광주", "대전광역시": "대전",
    "울산광역시": "울산", "세종특별자치시": "세종", "경기도": "경기",
    "강원도": "강원", "강원특별자치도": "강원", "충청북도": "충북",
    "충청남도": "충남", "전라북도": "전북", "전북특별자치도": "전북",
    "전라남도": "전남", "경상북도": "경북", "경상남도": "경남",
    "제주특별자치도": "제주",
}

def wide_to_long(df, region_col, value_col):
    df = df.copy()
    df.columns = df.columns.map(lambda x: str(x).strip())
    df["지역"] = df[region_col].astype(str).str.strip().map(region_map)

    if df["지역"].isna().any():
        raise ValueError(f"{value_col}: 미등록 지역명이 있습니다.")
    if set(df["지역"]) != set(region_order) or df["지역"].duplicated().any():
        raise ValueError(f"{value_col}: 지역 누락 또는 중복이 있습니다.")

    result = df[["지역", *years]].melt(
        id_vars="지역", var_name="연도", value_name=value_col
    )
    result["연도"] = result["연도"].astype(int)
    result[value_col] = pd.to_numeric(result[value_col], errors="raise")
    return result

capacity_raw = pd.read_excel(childcare_paths["numerator_active"])

population_raw = pd.read_excel(childcare_paths["denominator"])
population_raw.columns = population_raw.columns.map(lambda x: str(x).strip())
population_raw = population_raw.rename(columns={f"{y} 년": y for y in years})

for col in ["행정구역(시군구)별", "연령별", "항목"]:
    population_raw[col] = population_raw[col].astype(str).str.strip()

population_selected = population_raw[
    population_raw["행정구역(시군구)별"].isin(region_map)
    & population_raw["연령별"].isin([f"{age}세" for age in range(5)])
    & population_raw["항목"].eq("총인구수[명]")
].copy()

population_selected["지역"] = population_selected["행정구역(시군구)별"].map(region_map)

age_counts = population_selected.groupby("지역").size().reindex(region_order, fill_value=0)
if not age_counts.eq(5).all():
    raise ValueError(f"지역별 0–4세 자료 오류:\n{age_counts[age_counts.ne(5)]}")

population_selected[years] = population_selected[years].apply(pd.to_numeric, errors="raise")
population_wide = population_selected.groupby("지역", as_index=False)[years].sum()

childcare_comparison = (
    wide_to_long(capacity_raw, "시도별", "어린이집_정원")
    .merge(
        wide_to_long(facility_count_raw, "시도별", "어린이집_수"),
        on=["지역", "연도"], validate="one_to_one"
    )
    .merge(
        wide_to_long(population_wide, "지역", "0_4세_인구"),
        on=["지역", "연도"], validate="one_to_one"
    )
)

order_map = {r: i for i, r in enumerate(region_order)}
childcare_comparison["_순서"] = childcare_comparison["지역"].map(order_map)
childcare_comparison = (
    childcare_comparison.sort_values(["_순서", "연도"])
    .drop(columns="_순서")
    .reset_index(drop=True)
)

if len(childcare_comparison) != 180 or childcare_comparison.isna().any().any():
    raise ValueError("결합 결과에 행 수 오류 또는 결측값이 있습니다.")

print(f"결합 결과: {childcare_comparison.shape}")
display(childcare_comparison.head(12))

결합 결과: (180, 5)


,지역,연도,어린이집_정원,어린이집_수,0_4세_인구
0,전국,2016,1767224,41084,2204271
1,전국,2017,1756603,40238,2079115
2,전국,2018,1732324,39171,1974244
3,전국,2019,1686873,37371,1845122
4,전국,2020,1628260,35352,1677023
5,전국,2021,1557496,33246,1530469
6,전국,2022,1476686,30923,1427590
7,전국,2023,1401440,28954,1334588
8,전국,2024,1340087,27387,1271776
9,전국,2025,1285032,26064,1254501


## 6. 기존 산식 오류 검증

두 산식을 같은 지역·연도 자료로 계산한다.

- 정원 기준: `어린이집 정원 ÷ 0–4세 인구 × 100`
- 개수 기준: `어린이집 수 ÷ 0–4세 인구 × 100`

기존 가공값과 개수 기준 값의 절대오차를 계산한다. `기존값 최대 오차: 0.0`은 180개 지역·연도 값 모두가 개수 기준 산식과 정확히 일치한다는 뜻이며, 기존 산식 오류가 확정된다.

In [22]:
# cell 13

childcare_comparison["정원_기준_보급률"] = (
    childcare_comparison["어린이집_정원"]
    / childcare_comparison["0_4세_인구"] * 100
)
childcare_comparison["개수_기준_보급률"] = (
    childcare_comparison["어린이집_수"]
    / childcare_comparison["0_4세_인구"] * 100
)

existing_raw = pd.read_excel(
    childcare_paths["processed"],
    sheet_name="2-1.1. 보육시설 보급률"
)
existing_rate = wide_to_long(existing_raw, "지역", "기존_보급률")

childcare_comparison = childcare_comparison.merge(
    existing_rate,
    on=["지역", "연도"],
    validate="one_to_one"
)
childcare_comparison["기존값_오차"] = (
    childcare_comparison["기존_보급률"]
    - childcare_comparison["개수_기준_보급률"]
).abs()

print("기존값 최대 오차:", childcare_comparison["기존값_오차"].max())
display(childcare_comparison.head(12).round(6))

기존값 최대 오차: 0.0


,지역,연도,어린이집_정원,어린이집_수,0_4세_인구,정원_기준_보급률,개수_기준_보급률,기존_보급률,기존값_오차
0,전국,2016,1767224,41084,2204271,80.172719,1.863836,1.863836,0.0
1,전국,2017,1756603,40238,2079115,84.488015,1.935343,1.935343,0.0
2,전국,2018,1732324,39171,1974244,87.746196,1.984101,1.984101,0.0
3,전국,2019,1686873,37371,1845122,91.423386,2.025395,2.025395,0.0
4,전국,2020,1628260,35352,1677023,97.092288,2.108021,2.108021,0.0
5,전국,2021,1557496,33246,1530469,101.765929,2.172275,2.172275,0.0
6,전국,2022,1476686,30923,1427590,103.439083,2.166098,2.166098,0.0
7,전국,2023,1401440,28954,1334588,105.009186,2.169508,2.169508,0.0
8,전국,2024,1340087,27387,1271776,105.371308,2.153445,2.153445,0.0
9,전국,2025,1285032,26064,1254501,102.433717,2.077639,2.077639,0.0


## 7. 수정값 표 생성과 인계 상태

`정원_기준_보급률`을 기존 가공 파일과 같은 `지역 × 연도` 형태로 변환해 `childcare_corrected`에 저장한다. 정상 결과는 18행 × 11열이며 결측값이 없어야 한다.

**현재 완료:** 보육시설 보급률의 원자료 연결, 기존 오류 입증, 2016–2025년 수정값 생성.

**후속 작업:** 수정값을 실제 가공 파일·통합본에 반영하고 일치 여부를 재검증한 뒤, 나머지 구조환경지표의 산식 검증을 이어간다.

In [24]:
# cell 14: 수정된 보육시설 보급률 표 생성

childcare_corrected = (
    childcare_comparison.pivot(
        index="지역", columns="연도", values="정원_기준_보급률"
    )
    .reindex(index=region_order, columns=range(2016, 2026))
    .reset_index()
)

if childcare_corrected.shape != (18, 11) or childcare_corrected.isna().any().any():
    raise ValueError("수정 결과의 지역·연도 또는 결측값을 확인하세요.")

display(childcare_corrected.round(6))

연도,지역,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,전국,80.172719,84.488015,87.746196,91.423386,97.092288,101.765929,103.439083,105.009186,105.371308,102.433717
1,서울,71.760207,76.539425,81.103646,84.620627,90.898436,96.378477,98.246631,99.170947,99.913749,96.602131
2,부산,67.900094,71.934077,75.343530,80.169132,86.273543,91.421541,93.692785,94.970807,93.110380,88.705587
3,대구,75.863425,79.655342,81.366508,83.797264,89.928571,93.622376,97.079481,98.046434,98.308853,95.607761
4,인천,72.337782,77.077966,81.379203,85.178092,91.364203,95.631746,94.561012,95.323432,94.397415,90.429878
5,광주,95.659757,103.081292,105.433751,107.090410,112.815255,109.276941,109.487282,111.519683,113.663065,112.273607
6,대전,79.918489,84.948023,86.960597,89.703130,94.706820,97.009501,94.069161,91.505374,91.449260,89.494949
7,울산,69.930689,74.560186,78.919881,85.237655,90.700375,94.685686,96.302139,98.812328,98.323008,95.434192
8,세종,73.850035,76.240508,84.151680,87.735249,93.450074,97.494749,99.077199,108.238983,112.987013,109.152997
9,경기,78.161356,81.743124,84.387387,88.428539,93.587910,97.861527,99.529469,102.370861,102.568124,99.795011
